# Planning and Reasoning for Agents

**Prerequisites:** Python functions, lists, and dictionaries. **Estimated time:** 35 minutes.

**Objectives:** Represent a goal as explicit subgoals; check preconditions before an action; compare a fixed plan with replanning after an observation.

## Concept recap and assumptions

Planning searches over possible action sequences. This lesson uses a tiny deterministic delivery world: actions have preconditions, effects, and unit costs. The planner is complete only within its bounded state space; it does not establish that a real-world action is safe.

## 1. Transparent local baseline

Run this deterministic, dependency-free baseline. All records are synthetic, and the code has no network access or side effects.

In [ ]:
import random
random.seed(20260922)
from collections import deque

# State is (at_depot, package_loaded, delivered). Every transition is local.
actions = {
    'load': lambda s: (s[0], True, s[2]) if s[0] and not s[1] else None,
    'drive': lambda s: (False, s[1], s[2]) if s[0] and s[1] else None,
    'deliver': lambda s: (s[0], s[1], True) if not s[0] and s[1] else None,
}
def shortest_plan(start, goal):
    queue = deque([(start, [])]); seen = {start}
    while queue:
        state, plan = queue.popleft()
        if goal(state): return plan
        for name, transition in actions.items():
            nxt = transition(state)
            if nxt is not None and nxt not in seen:
                seen.add(nxt); queue.append((nxt, plan + [name]))
    return None

start = (True, False, False)
plan = shortest_plan(start, lambda s: s[2])
print(plan)
assert plan == ['load', 'drive', 'deliver']

## 2. Extension / realistic constraint

Add a fuel precondition to `drive`, then make the planner report failure when fuel is unavailable. A planner should expose why no plan exists rather than silently inventing an action.

## 3. Comparison and evaluation

Breadth-first search returns the minimum number of actions for this unit-cost finite graph. Its memory and time are O(|S| + |E|); the state space grows exponentially with independent facts.

## 4. Edge and failure cases

Try a goal already true (empty plan) and a goal unreachable under the action preconditions. Add a maximum expansion budget and distinguish exhausted search from proven impossibility.

## 5. Exercises

Implement a `charge` action at the depot and a one-bit fuel state. Tests: no fuel means no delivery plan; charging before loading yields a valid plan. Explain how non-unit action costs change the search algorithm.

## Summary and next practice

The planner found a shortest valid action sequence because preconditions constrained the transition graph. Real systems need state freshness, cost models, and authorization checks before executing a plan.

Related practice: agentic-ai/interview-prep/agentic-ai-interview-questions.md — planning, state representation, and replanning questions.